In [0]:
from pyspark.sql import functions as F

In [0]:
gold_country_daily = spark.sql("""
SELECT
    EventDate,
    ActionGeo_CountryCode AS CountryCode,

    COUNT(*) AS EventCount,
    SUM(NumMentions) AS TotalMentions,
    SUM(NumSources) AS TotalSources,
    SUM(NumArticles) AS TotalArticles,

    AVG(GoldsteinScale) AS AvgGoldsteinScale,
    AVG(AvgTone) AS AvgTone

FROM workspace.silver.events

WHERE ActionGeo_CountryCode IS NOT NULL

GROUP BY
    EventDate,
    ActionGeo_CountryCode
""")

display(
    gold_country_daily
    .orderBy(
        F.col("EventDate").desc(),
        F.col("EventCount").desc()
    )
    .limit(20)
)

In [0]:
(
    gold_country_daily.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.country_daily")
)

In [0]:
display(
    spark.sql("""
        SELECT
            CountryCode,
            SUM(EventCount) AS TotalEvents
        FROM workspace.gold.country_daily
        GROUP BY CountryCode
        ORDER BY TotalEvents DESC
        LIMIT 20
    """)
)

In [0]:
gold_category_daily = spark.sql("""
SELECT
    EventDate,
    EventRootCode,

    COUNT(*) AS EventCount,
    SUM(NumMentions) AS TotalMentions,
    SUM(NumSources) AS TotalSources,
    SUM(NumArticles) AS TotalArticles,
    AVG(GoldsteinScale) AS AvgGoldsteinScale,
    AVG(AvgTone) AS AvgTone

FROM workspace.silver.events

WHERE EventRootCode IS NOT NULL

GROUP BY
    EventDate,
    EventRootCode
""")

display(
    gold_category_daily
    .orderBy(F.col("EventDate").desc(), F.col("EventCount").desc())
    .limit(20)
)

In [0]:
(
    gold_category_daily.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.category_daily")
)